# L1b: Toolchain and Notebook Smoke Test

This lab verifies the complete course workflow: activate the shared Julia environment, load local course code, run a package function, visualize generated data, and execute tests.

> **Learning Objectives**
> 1. Run a Julia notebook from top to bottom in the course environment.
> 2. Explain what the meeting-local `Include.jl` contributes.
> 3. Confirm that external packages and local source code are available.
> 4. Read a small `Test` test set as an executable specification.

---

## Setup, Data, and Prerequisites

In our case, our `Include.jl` file will set paths so our notebook knows where to find things, and then will load external packages into the global scope with [the `using` command](https://docs.julialang.org/en/v1/base/base/#using). This makes the content of the package visible to us. 

In [ ]:
include(joinpath(@__DIR__, "Include.jl"))

Now that we have our environment setup, we can do some stuff.

## Task 1: Let's build and visualize a Normal Distribution
In this task, let's test our installation by sampling a model of [a Normal probability distribution](https://en.wikipedia.org/wiki/Normal_distribution), and then visualizing the samples. First, let's draw samples from the distribution and save them in the `samples::Array{Float64,1}` array. 

> __What is the `let` block?__ The [`let` block](https://docs.julialang.org/en/v1/base/base/#let) creates a new hard scope and optionally introduces new local bindings. Variables introduced inside a `let` block are local to that block and don't affect variables of the same name in the outer scope. In our case, the `let` block allows us to create local variables (`number_of_samples` and the local `samples`) that are only visible within the block, while the final value of `samples` is returned and assigned to the global variable `samples`. This is a common Julia pattern for organizing code and avoiding namespace pollution.

We use [the built-in `randn(...)` method](https://docs.julialang.org/en/v1.11/stdlib/Random/#Base.randn) to generate samples from a standard normal distribution (mean 0, variance 1).

In [ ]:
samples = let

    # initialize -
    number_of_samples = 10000; # set the number of samples we want to generate
    samples = randn(number_of_samples);

    samples; # return
end;

Now, let's plot the `sample::Array{Float64,1}` array using [the `histogram(...)` method exported by the `UnicodePlots.jl` package](https://juliaplots.org/UnicodePlots.jl/dev/api/#UnicodePlots.histogram-Tuple%7BAbstractArray%7D). 

In [ ]:
let

    # initialize -
    data = samples; # random array
    number_of_bins = 20; # how many bins?
    vertical = false; # vertical or horizontal?
    closed = :left; # which side of the interval is closed?

    # make the histogram -
    histogram(data, nbins=number_of_bins, vertical = vertical, closed = closed); 
end

In [ ]:
do_you_see_the_histogram = false; # TODO: set this to true once you actually see the histogram above

## Task 2: Can we see code that we wrote?
In this task, we check that methods that we wrote (that were included when we called the `Include.jl` file) are now visible. In [the `HelloWorld.jl` file](src/HelloWorld.jl), we defined a simple method called `printgreeting()` that __returns__ the string `"Hello World!"`.

> __Return, not print:__ despite the name, `printgreeting()` has no side effect — it prints nothing. The notebook shows the string because the cell _displays the value of its last expression_. That distinction matters as soon as you start composing functions: a printed value is gone, a returned value can be used.

We'll save the returned value in the `message_that_we_get::String` variable:

In [ ]:
message_that_we_get = printgreeting() # returns "Hello World!"; the notebook displays it

## Tests
In the code block below, we check some values in your notebook and give you feedback on which items are correct or different. `Unhide` the code block below (if you are curious) about how we implemented the tests and what we are testing.

In [ ]:
let

    @testset verbose = true "CHEME 4/5800 L1b Test Suite" begin
        
        # Test 1: Environment Setup
        @testset "Environment Setup Tests" begin
            @test isdefined(Main, :CHEME5800_L1B_ROOT)
            @test CHEME5800_L1B_ROOT == @__DIR__
            @test @isdefined printgreeting
        end
        
        # Test 2: Sample Generation Tests
        @testset "Sample Generation Tests" begin
            @test @isdefined samples
            @test isa(samples, Array{Float64,1})
            @test length(samples) == 10000
            @test !isempty(samples)
            
            # Statistical properties of normal distribution
            sample_mean = Statistics.mean(samples)
            sample_std = Statistics.std(samples)
            @test abs(sample_mean) < 0.1
            @test abs(sample_std - 1.0) < 0.1
        end
        
        # Test 3: Histogram Flag Tests
        # This one fails until you look at the plot and flip the flag yourself.
        @testset "Histogram Flag Tests" begin
            @test @isdefined do_you_see_the_histogram
            @test isa(do_you_see_the_histogram, Bool)
            @test do_you_see_the_histogram == true
        end
        
        # Test 4: HelloWorld Function Tests
        @testset "HelloWorld Function Tests" begin
            @test @isdefined message_that_we_get
            @test isa(message_that_we_get, String)
            @test message_that_we_get == "Hello World!"
            
            # Test the function directly
            greeting = printgreeting()
            @test greeting == "Hello World!"
            @test isa(greeting, String)
        end
        
        # Test 5: Data Type and Structure Tests
        @testset "Data Type and Structure Tests" begin
            # Test that all variables have expected types
            @test isa(samples, Vector)
            @test eltype(samples) == Float64
            
            # Test that samples are finite (no NaN or Inf values)
            @test all(isfinite.(samples))
            
            # Test range - normal distribution should have most values within ±4 standard deviations
            extreme_values = count(abs.(samples) .> 4.0)
            @test extreme_values < 100
        end
        
        # Test 6: Package and Function Availability Tests
        @testset "Package and Function Availability Tests" begin
            @test @isdefined histogram
            @test @isdefined randn
        end
    end
end;

## Summary

> **Key Takeaways**
> 1. One local include connects the notebook to the pinned root environment.
> 2. A smoke test should exercise packages, local source, computation, and tests.
> 3. A successful run establishes a known-good baseline for later debugging.

---